# 🧬 A Gentle Introduction to Analysing Your Data with Python
**Workshop — Konstantinos Kalaitzidis**

---

We will walk through an example of a data analysis pipeline for inspecting biological data. 
By the end of the session you will have loaded, cleaned, explored, and visualised a real biological dataset in Python.

**What we'll cover:**
1. Jupyter Notebook Tips & Tricks
2. Loading & Inspecting Data
3. Cleaning & Wrangling
4. Exploratory Statistics
5. Visualisation

## How to use this notebook

Each section has:
- A **narrative cell** (like this one) explaining the concept
- A **code cell** below that you run with `Shift + Enter`

You have your own copy — feel free to edit, break things, and experiment.
That is the best way to learn.

>If something breaks: `Runtime → Restart and run all` to start fresh.


## The Data Analysis Pipeline

Every project follows roughly the same steps:

```
Raw Data → Load → Inspect → Clean → Explore → Visualise → Model
```

Today's dataset: **clinical measurements from breast tumor biopsies**
(Breast Cancer Wisconsin Dataset).

- **569 patient samples**, 30 numerical features per sample
- Features: cell radius, texture, perimeter, area, smoothness, concavity…
- Target: `diagnosis` — Malignant or Benign

📄 **Dataset resources:**
- [UCI ML Repository](https://archive.ics.uci.edu/dataset/17/breast+cancer+wisconsin+diagnostic)
- [scikit-learn documentation](https://scikit-learn.org/stable/modules/generated/sklearn.datasets.load_breast_cancer.html)

---
## 1. 💻 Getting Started with Google Colab

Google Colab is a free, cloud-based environment for running Python notebooks
directly in your browser — no installation needed.

**The golden rule: run cells in order, from top to bottom.**
Each cell may depend on the ones above it. Skipping a cell or running
things out of order is the most common cause of errors.

---

### Running cells

| Action | How |
|---|---|
| Run a cell and move to the next | `Shift + Enter` |
| Run a cell and stay | `Ctrl + Enter` |
| Run all cells from top to bottom | `Runtime → Run all` |
| Restart and run all (fix most problems) | `Runtime → Restart and run all` |

> 💡 If something breaks or behaves unexpectedly, `Runtime → Restart and run all`
> fixes 90% of problems. It clears all variables and re-runs the notebook fresh.

---

### Your copy is yours

Because you clicked **"Copy to Drive"**, this is your personal copy.
Whatever you run, edit, or break only affects your notebook —
not anyone else's, and not the original.

---

### Useful shortcuts

| Shortcut | What it does |
|---|---|
| `Tab` | Autocomplete — shows available methods and variables |
| `Shift + Tab` | Show documentation for a function |
| `Ctrl + /` | Comment or uncomment a line |

---

### A note on sessions

Colab disconnects after **~90 minutes of inactivity**. If you step away and
come back, you may find the runtime has reset — all variables will be gone.
The fix is always `Runtime → Restart and run all`.

In [ ]:
# Use ? to pull up documentation for any function
# Run this cell and a help panel will appear at the bottom
import pandas as pd
pd.read_csv?


---
## 2. Loading & Inspecting Data

The first step in any analysis: load your data and understand its structure.

**pandas** 🐼 is the go-to Python library for tabular data.
Think of it as a supercharged spreadsheet you control with code.

A pandas **DataFrame** is just a table — rows are samples, columns are features.


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_breast_cancer

%matplotlib inline
sns.set_theme(style="whitegrid")

print("✅ All libraries loaded!")


In [ ]:
# ── Load the dataset ─────────────────────────────────────────────────────
raw = load_breast_cancer()

# Convert to a pandas DataFrame
df = pd.DataFrame(raw.data, columns=raw.feature_names)

# Add the target column  (0 = malignant, 1 = benign)
df['diagnosis'] = raw.target
df['diagnosis_label'] = df['diagnosis'].map({0: 'malignant', 1: 'benign'})

print(f"Dataset loaded: {df.shape[0]} samples, {df.shape[1]} columns")


In [ ]:
# First look — always start here
df.head()


In [ ]:
# How big is the dataset?
print("Shape (rows, columns):", df.shape)
print()

# What are the columns and their data types?
print("Column types:")
print(df.dtypes)


In [ ]:
# How balanced is our target?
print("Diagnosis distribution:")
print(df['diagnosis_label'].value_counts())
print()
print(df['diagnosis_label'].value_counts(normalize=True).round(2))


---
## 3. Data Cleaning

Real-world data is messy. You will almost always encounter:
- **Missing values** (NaN)
- **Wrong data types**
- **Inconsistent labels or column names**

Let's simulate a realistic scenario by introducing missing values,
then handle them properly.

**Key functions:**

| Function | What it does |
|---|---|
| `df.isnull().sum()` | Count missing values per column |
| `df.fillna(value)` | Fill missing values |
| `df.dropna()` | Drop rows with missing values |
| `df.rename(columns={...})` | Rename columns |
| `df.loc[rows, cols]` | Select by label |
| `df.iloc[rows, cols]` | Select by position |


In [ ]:
# Simulate missing data — as you'd find in a real clinical dataset
np.random.seed(42)
df_messy = df.copy()

# Randomly set 20 values in 'mean radius' to NaN
idx1 = np.random.choice(df_messy.index, size=20, replace=False)
df_messy.loc[idx1, 'mean radius'] = np.nan

# And 10 values in 'mean texture'
idx2 = np.random.choice(df_messy.index, size=10, replace=False)
df_messy.loc[idx2, 'mean texture'] = np.nan

print("Missing values introduced. Let's find them...")


In [ ]:
# Check for missing values across all columns
missing = df_messy.isnull().sum()

# Only show columns that actually have missing values
print("Columns with missing values:")
print(missing[missing > 0])


In [ ]:
# How would you fill the missing values in 'mean radius'?
# A common approach: fill with the column mean.

# ── Solution ─────────────────────────────────────────────────────────────
df_messy['mean radius']  = df_messy['mean radius'].fillna(df_messy['mean radius'].mean())
df_messy['mean texture'] = df_messy['mean texture'].fillna(df_messy['mean texture'].mean())

# Verify
remaining = df_messy.isnull().sum().sum()
print(f"Missing values remaining: {remaining}")
print("✅ Dataset is clean!")


In [ ]:
# Renaming columns for readability
df_clean = df_messy.rename(columns={
    'mean radius':   'radius',
    'mean texture':  'texture',
    'mean area':     'area',
    'mean smoothness': 'smoothness'
})

# Selecting a subset of columns with loc
subset_cols = ['radius', 'texture', 'area', 'smoothness', 'diagnosis_label']
df_subset = df_clean.loc[:, subset_cols]
df_subset.head()


In [ ]:
# Filtering rows — e.g. only malignant samples
malignant = df[df['diagnosis_label'] == 'malignant']
benign    = df[df['diagnosis_label'] == 'benign']

print(f"Malignant samples : {len(malignant)}")
print(f"Benign samples    : {len(benign)}")


---
## 4. Exploratory Statistics

Before building any model, we explore the data to develop intuition:

- What is the range of values?
- Are there differences between groups?
- Which features are correlated with each other?

**Key functions:**

| Function | What it does |
|---|---|
| `df.describe()` | Summary statistics (min, max, mean, etc) |
| `df.groupby(col).mean()` | Compare group averages |
| `df.corr()` | Pairwise correlations between all features |


In [ ]:
# Summary statistics for all numeric columns
df.describe().round(2) 

In [ ]:
# Compare feature means between malignant and benign tumors
features_to_compare = ['mean radius', 'mean texture', 'mean area',
                       'mean smoothness', 'mean concavity']

group_means = df.groupby('diagnosis_label')[features_to_compare].mean()
print("Mean feature values by diagnosis:")
group_means.round(3)

In [ ]:
# Correlation matrix — which features move together?
selected = ['mean radius', 'mean texture', 'mean area',
            'mean smoothness', 'mean concavity', 'mean symmetry', 'diagnosis']

corr_matrix = df[selected].corr().round(2)
corr_matrix

---
## 5. Visualisation

We'll use two libraries:
- **matplotlib** — the foundational Python plotting library 
- **seaborn** — a higher-level library built on top of matplotlib 

**Plots we'll make:**
1. **Histogram** — distribution of a single feature across all samples. Useful for spotting skew, outliers, or whether two groups overlap.
2. **Boxplot** — great for comparing groups side by side. Summarises a distribution across groups using five numbers: minimum, lower quartile, median, upper quartile, maximum. 
3. **Scatterplot** — plots the relationship between two features. Reveals clusters and whether groups are separable in two dimensions.
4. **Heatmap** — visualises the full correlation matrix at a glance such as which features move together, and which are independent.



In [ ]:
# ── 1. Histogram ─────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(benign['mean radius'],    bins=30, alpha=0.6, label='Benign',    color='steelblue')
ax.hist(malignant['mean radius'], bins=30, alpha=0.6, label='Malignant', color='salmon')

ax.set_xlabel('Mean Radius')
ax.set_ylabel('Count')
ax.set_title('Distribution of Mean Radius by Diagnosis')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── 2. Boxplot ───────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

sns.boxplot(
    data=df, x='diagnosis_label', y='mean area',
    palette={'benign': 'steelblue', 'malignant': 'salmon'},
    ax=ax
)

ax.set_title('Tumor Area by Diagnosis')
ax.set_xlabel('Diagnosis')
ax.set_ylabel('Mean Area')
plt.tight_layout()
plt.show()

In [ ]:
# ── 3. Scatterplot ───────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))

sns.scatterplot(
    data=df, x='mean radius', y='mean concavity',
    hue='diagnosis_label',
    palette={'benign': 'steelblue', 'malignant': 'salmon'},
    alpha=0.7, ax=ax
)

ax.set_title('Mean Radius vs Mean Concavity')
ax.set_xlabel('Mean Radius')
ax.set_ylabel('Mean Concavity')
plt.tight_layout()
plt.show()

In [ ]:
# ── 4. Correlation Heatmap ───────────────────────────────────────────────
selected = ['mean radius', 'mean texture', 'mean area',
            'mean smoothness', 'mean concavity', 'mean symmetry', 'diagnosis']

corr = df[selected].corr()

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, square=True, linewidths=0.5, ax=ax)

ax.set_title('Feature Correlation Heatmap')
plt.tight_layout()
plt.show()

## 🔍 So what have we learned?

**1. Malignant tumors are physically larger.**
Every size-related feature — radius, perimeter, area — is consistently higher in malignant samples. The histogram and boxplot both showed this clearly: the distributions are shifted, though not completely separated.

**2. No single feature is enough.**
The histogram showed overlap between the two groups even for the most informative features. You cannot reliably classify a tumor from one measurement alone. This is precisely why we have 30 features and why machine learning is useful here.

**3. Many features are highly correlated with each other.**
The heatmap revealed that radius, perimeter, and area are almost perfectly correlated. They all measure size in slightly different ways. In a production model, keeping all three adds noise without adding information. This is called multicollinearity. For today this is fine, but it is something to be aware of.

**4. The two classes are partially separable in 2D.**
The scatterplot of radius vs concavity showed two distinct — if overlapping — clouds. This is an encouraging sign: if groups are visually separable in just two dimensions, a model using all 30 features should do considerably better.

> 💡 Exploratory analysis is not about producing final results,
> but to build enough intuition about your data to make informed decisions
> about how to model it.

---
## Wrap-Up

Congratulations for making it till here!

| Step | Key functions |
|---|---|
| Load & Inspect | `pd.DataFrame`, `head()`, `shape`, `dtypes`, `value_counts()` |
| Clean & Wrangle | `isnull()`, `fillna()`, `dropna()`, `rename()`, `loc[]` |
| Explore | `describe()`, `groupby()`, `corr()` |
| Visualise | histograms, boxplots, scatterplots, heatmaps |

**Ask yourself**:
- What kind of data do I work with, and which of these steps would be most relevant for it?
- Are there missing values in my dataset, and if so, why are they missing. Is it random, or does it mean something?
- Which features in my data are likely to be correlated, and does that affect how I should interpret them?
- Which type of plots should I generate to visualise and understand my dataset, why, and what would I hope to see?
- What do I think I understand after reflecting on these plots?
- How do I proceed with answering my research question based on this analysis?



**Where to go from here:**

*Python fundamentals*
- 🐍 [NAISS Intro to Python](https://uppmax.github.io/naiss_intro_python/sessions/introduction_basic_python/) — a structured introduction to Python basics
- 📖 [Python Data Science Handbook](https://jakevdp.github.io/PythonDataScienceHandbook/) — a free, comprehensive book covering NumPy, pandas, matplotlib, and scikit-learn in depth

*pandas*
- 🐼 [10 Minutes to pandas](https://pandas.pydata.org/pandas-docs/stable/user_guide/10min.html) — the official quick-start guide
- 📄 [pandas Cheat Sheet](https://pandas.pydata.org/Pandas_Cheat_Sheet.pdf) — a one-page reference for the most common operations 

*Visualisation*
- 🎨 [seaborn example gallery](https://seaborn.pydata.org/examples/index.html)
- 🏆 [Kaggle Learn — free micro-courses](https://www.kaggle.com/learn)


---
*Questions? Open an issue on the GitHub repository or reach out at kon.kalaitzidis@scilifelab.se


## 6. Self-Work Exercises

These exercises are for you to work through at your own pace.
Each one is a small extension of something we already did together
so you have all the tools you need. A solution is hidden beneath each exercise.
Try to solve it yourself first before revealing it.

---

### Exercise 1 — A New Histogram

We plotted the distribution of `mean radius` split by diagnosis.
Now do the same for **`mean concavity`**.

- Does the separation between malignant and benign look stronger or weaker than for radius?
- What does this tell you about how useful concavity might be as a feature?

<details>
<summary>💡 Click to reveal solution</summary>

```python
fig, ax = plt.subplots(figsize=(8, 4))

ax.hist(benign['mean concavity'],    bins=30, alpha=0.6, label='Benign',    color='steelblue')
ax.hist(malignant['mean concavity'], bins=30, alpha=0.6, label='Malignant', color='salmon')

ax.set_xlabel('Mean Concavity')
ax.set_ylabel('Count')
ax.set_title('Distribution of Mean Concavity by Diagnosis')
ax.legend()
plt.tight_layout()
plt.show()

```

You should see that the separation between groups is actually stronger for concavity than for radius —
malignant tumors have markedly higher concavity values with less overlap.
This is consistent with what the correlation heatmap suggested.

</details>

---


In [ ]:
# Exercise 1 — your code here



### Exercise 2 — Side-by-Side Boxplots

We used a boxplot to compare `mean area` between groups.
Now do the same for **`mean texture`** and **`mean concavity`** — but this time,
plot them **side by side** in a single figure with two subplots.

- Which feature shows a cleaner separation between groups?
- Does this match what the heatmap suggested?

> 💡 Hint: `fig, axes = plt.subplots(1, 2, figsize=(12, 5))` creates a figure with
> two side-by-side panels. Each panel is accessed as `axes[0]` and `axes[1]`.

<details>
<summary>💡 Click to reveal solution</summary>

```python
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(
    data=df, x='diagnosis_label', y='mean texture',
    palette={'benign': 'steelblue', 'malignant': 'salmon'},
    ax=axes[0]
)
axes[0].set_title('Mean Texture by Diagnosis')
axes[0].set_xlabel('Diagnosis')
axes[0].set_ylabel('Mean Texture')

sns.boxplot(
    data=df, x='diagnosis_label', y='mean concavity',
    palette={'benign': 'steelblue', 'malignant': 'salmon'},
    ax=axes[1]
)
axes[1].set_title('Mean Concavity by Diagnosis')
axes[1].set_xlabel('Diagnosis')
axes[1].set_ylabel('Mean Concavity')

plt.tight_layout()
plt.show()

```

Concavity shows a much cleaner separation than texture — the boxes barely overlap,
whereas texture has considerable overlap between groups.
This is consistent with concavity's stronger correlation with diagnosis in the heatmap.

</details>

---


In [ ]:
# Exercise 2 — your code here



### Exercise 3 — Mean and Spread by Group

We used `groupby().mean()` to compare average feature values between groups.
Now extend this for **`mean concavity`** and **`mean symmetry`** by computing
both the **mean** and the **standard deviation** for each diagnosis group.

- Does one group show more variability than the other?
- What might that tell you biologically?

> 💡 Hint: you can chain `.agg(['mean', 'std'])` instead of `.mean()` alone
> to get both statistics in one table.

<details>
<summary>💡 Click to reveal solution</summary>

```python
features = ['mean concavity', 'mean symmetry']

summary = df.groupby('diagnosis_label')[features].agg(['mean', 'std']).round(3)

print("Mean and standard deviation by diagnosis group:")
print(summary)
```

Malignant tumors show higher variability in concavity — a larger standard deviation
means their cell boundaries are not only more irregular on average,
but also more inconsistent from patient to patient.
Benign tumors cluster more tightly around lower concavity values.
This kind of variability is itself a biological signal worth paying attention to.

</details>

---


In [ ]:
# Exercise 3 — your code here
